上下文编辑中间件，该中间件提供了上下文管理的一种方式。
1. ContextEditingMiddleware 的价值：大模型多轮对话时，如果频繁调用产生大量文本的工具
（如代码执行、网页爬取），历史记录会急剧膨胀。这个中间件就像一个“上下文抽脂手术”，在不
影响当前对话的前提下，自动在后台删掉之前沉淀的工具调用废话，从而极大地节省 Token 费用
并防止超出模型最大上下文窗口（Context Window）。
2. InMemorySaver ：它在内存中开辟了一个空间。第二轮和第三轮提问时，Agent 能通过
thread_id 自动找回前几轮的记忆。

1. 实验组-启用上下文编辑

In [1]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import ContextEditingMiddleware,ClearToolUsesEdit
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key = os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
    f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=main_model,
    tools=[get_weather],
    middleware=[
        ContextEditingMiddleware(
            edits=[
                ClearToolUsesEdit(
                    trigger=50,
                    keep=0,
                )
            ]
        ),
    ],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = RunnableConfig(
    configurable={
        "thread_id": "1"
    },
)

for i in range(3):
    print("=" * 40, f"当前是第{i + 1}轮调用", "=" * 40)
    count +=1
    res = agent.invoke({
        "messages": [HumanMessage(f"当前是第{i + 1}次询问： 今天深圳天气如何，一句话回答")]
    }, config=config)

    print("-------- 本次返回的 messages --------")
    for msg in res["messages"]:
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")



======================================== 当前是第1轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 328, 'output_tokens': 6, 'total_tokens': 334, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}
======================================== 当前是第2轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 328, 'output_tokens': 6, 'total_tokens': 334, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}
本次token用量：{'input_tokens': 447, 'output_tokens': 6, 'total_tokens': 453, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}
======================================== 当前是第3轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 328, 'output_tokens': 6, 'total_tokens': 334, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 0}}


2. 对照组-不裁剪上下文

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool
from dotenv import load_dotenv
import os

load_dotenv()

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key = os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

count = 0

@tool
def get_weather(city: str):
    """查询指定城市天气"""
    global count
    return (f"当前是第 {count} 次调用工具，{city}今天天气晴朗"
    f"天气非常好，北风，非常适合出行，盼望着，盼望着，"
    f"春天来了。我喜欢春天，你喜欢吗，天气真的很不错"
    f"万里无云，天气晴朗，春和景明，哈哈哈哈哈哈，这是凑字数的"
    f"真不错，天气非常好，适合出行，这里token挺多的"
    f"可以出门玩，尅有跑步，钓鱼，爬山，一切都很好哈哈哈")

agent = create_agent(
    model=main_model,
    tools=[get_weather],
    checkpointer=InMemorySaver(),
)

config: RunnableConfig = RunnableConfig(
    configurable={
        "thread_id": "1"
    },
)

for i in range(3):
    print("=" * 40, f"当前是第{i + 1}轮调用", "=" * 40)
    count +=1
    res = agent.invoke({
        "messages": [HumanMessage(f"当前是第{i + 1}次询问： 今天深圳天气如何，一句话回答")]
    }, config=config)

    print("-------- 本次返回的 messages --------")
    for msg in res["messages"]:
        if isinstance(msg, AIMessage):
            if not msg.tool_calls:
                print(f"本次token用量：{msg.usage_metadata}")



======================================== 当前是第1轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 412, 'output_tokens': 217, 'total_tokens': 629, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 202}}
======================================== 当前是第2轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 412, 'output_tokens': 217, 'total_tokens': 629, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 202}}
本次token用量：{'input_tokens': 707, 'output_tokens': 83, 'total_tokens': 790, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 69}}
======================================== 当前是第3轮调用 ========================================
-------- 本次返回的 messages --------
本次token用量：{'input_tokens': 412, 'output_tokens': 217, 'total_tokens': 629, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reas